# Введение

## Цель работы
Изучение алгоритмов кластеризации, приобретение навыков оценки качества разбиения данных на кластеры и интерпретации результатов.

## Постановка задачи

1. Загрузить датасет для задачи кластеризации / классификации (например, с платформы Kaggle). Провести дескриптивный анализ данных: определить размерность, типы признаков, наличие пропусков. Оценить распределение переменных (близость к нормальному) с использованием визуализации (гистограммы). Проверить условие применения кластеризации: отсутствие классов, осмысленность кластеризации, отсутствие выбросов.

2. Выполнить стандартизацию / нормализацию числовых признаков. Обосновать выбор метода масштабирования. Построить матрицу диаграмм рассеивания для визуальной оценки структуры данных, предположительного количества кластеров и типа кластерной структуры. Аргументировать выбор методов кластеризации на основе формы, размера и плотности кластеров.

3. Реализовать кластеризацию двумя различными методами на выбор: K-means (K-средних); иерархическая кластеризация; DBSCAN; EM-алгоритм (Gaussian Mixture). Для методов, требующих задания числа кластеров (K-means, иерархическая), подобрать оптимальное значение k с использованием: метода локтя и / или анализа силуэта.

4. Рассчитать метрики качества для обоих методов: внутренние и внешние. Оценить расстояние между кластерами, внутрикластерные расстояния, компактность кластеров, центры кластеров. Опционально, если известно разделение на классы, посчитать: индекс Rand, индекс Жаккара и др. внешние метрики. Привести содержательную интерпретацию полученных значений.

5. Исследовать влияние параметров одного из методов (например, ε и min_samples для DBSCAN или количества кластеров k для K-means) на качество кластеризации.

6. Визуализировать полученные кластеры в пространстве признаков (использовать PCA для снижения размерности при необходимости). Проанализировать центры кластеров (для K-means) и дать содержательную интерпретацию выделенных групп. Сравнить результаты, полученные двумя разными методами.

# Описание датасета

Для работы выбран датасет [Industrial Equipment 🖥️Monitoring 🖲️Dataset](https://www.kaggle.com/datasets/dnkumars/industrial-equipment-monitoring-dataset). 

Этот датасет содержит симулированные данные, имитирующие мониторинг промышленного оборудования в реальном времени, включая турбины, компрессоры и насосы. Каждая строка датасета соответствует уникальному наблюдению и фиксирует ключевые параметры работы оборудования: температуру, давление, уровень вибрации и влажность. Также в набор данных включена информация о типе оборудования, месте его расположения и о том, считается ли оборудование неисправным.

Признаки:

| Название переменной | Тип переменной            | Описание                                                                                                      |
| ---------------     | ------------------------- | ------------------------------------------------------------------------------------------------------------- |
| **temperature**     | Числовая                  | Температура оборудования в момент наблюдения (°C). Характеризует тепловой режим работы.                       |
| **pressure**        | Числовая                  | Давление в системе в момент наблюдения (bar). Отражает рабочие условия и возможные отклонения.                |
| **vibration**       | Числовая                  | Уровень вибрации (нормированные единицы). Повышенные значения могут сигнализировать о механических проблемах. |
| **humidity**        | Числовая                  | Влажность окружающей среды в месте установки оборудования (%). Влияет на условия эксплуатации.                |
| **equipment**       | Категориальная            | Тип промышленного оборудования (например, Turbine, Compressor, Pump).                                         |
| **location**        | Категориальная            | Локация/город, где находится оборудование (например, Atlanta, Chicago и т.д.).                                |
| **faulty**          | Категориальная (бинарная) | Индикатор неисправности: 0 — исправно, 1 — неисправно. Используется для анализа/проверки кластеров.           |







In [ ]:
import pandas as pd
import math
import scipy.stats as stats
import matplotlib.pyplot as plt


df = pd.read_csv("equipment_anomaly_data.csv")
df.head()

: 

## Информация о датасете

In [ ]:
print("Размерность (rows, cols):", df.shape)

print("Число объектов (строк):", df.shape[0])
print("Число признаков (столбцов):", df.shape[1])

df.info()

Как мы видим в датасете отсутвуют пропуски. Признаки temperature, pressure, vibration, humidity являются числовыми, по ним мы и будем проводить кластеризацию. А признакники equipment, location, faulty - категориальные. При этом faulty явялется бинарным и имеет значения 0 или 1; equipment и location уже в виде текста.

## Дескриптивный анализ

In [ ]:
n = df.shape[0]
k = int(math.ceil(1 + math.log2(n)))

def interpret_pvalue(p):
     if p > 0.05:
         return "Нормальное"
     elif p > 0.01:
         return "Близко к нормальному"
     else:
         return "Далеко от нормального"
     
features = ["temperature", "pressure", "vibration", "humidity", "faulty"]
fig, axes = plt.subplots(1, 5, figsize=(25, 4))
axes = axes.flatten()

stat = {
    "Признак": [], "Среднее": [], "Медиана": [], "Мода": [], "Минимум": [], "Максимум": [],
    "Стандартное отклонение": [], "Дисперсия": [], "Асимметрия": [], "Эксцесс": [],
    "Оценка нормальности распредления (тест Колмагорова-Смирнова)": []
    }

for i, feature in enumerate(features):
    ks_test = stats.kstest(df[feature], 'norm', args=(df[feature].mean(), df[feature].std()))

    stat["Признак"].append(feature)
    stat["Среднее"].append(round(df[feature].mean(), 2))
    stat["Медиана"].append(round(df[feature].median(), 2))
    stat["Мода"].append(round(df[feature].mode()[0], 2))
    stat["Минимум"].append(round(df[feature].min(), 2))
    stat["Максимум"].append(round(df[feature].max(), 2))
    stat["Стандартное отклонение"].append(round(df[feature].std(), 2))
    stat["Дисперсия"].append(round(df[feature].var(), 2))
    stat["Асимметрия"].append(round(df[feature].skew(), 2))
    stat["Эксцесс"].append(round(df[feature].kurtosis(), 2))
    stat["Оценка нормальности распредления (тест Колмагорова-Смирнова)"].append(f"p={ks_test.pvalue:.3e} → {interpret_pvalue(ks_test.pvalue)}")


    k_ = k if feature != "quality" else 6
    axes[i].hist(df[feature], bins=k_, edgecolor='black', color='skyblue')
    axes[i].set_title(feature, fontsize=11)
    axes[i].set_xlabel("")
    axes[i].set_ylabel("Частота")

display(pd.DataFrame(stat))

По гистограммам и рассчитанным показателям (асимметрия, эксцесс, тест Колмогорова–Смирнова) видно, что все числовые признаки далеки от нормального распределения.

temperature: среднее 70.92, медиана 70.16 — центр примерно совпадает, но асимметрия 0.94 и высокий эксцесс 5.41 указывают на вытянутый правый хвост и наличие экстремальных значений. Тест Колмогорова–Смирнова даёт p≈2.95e−63, то есть нормальность уверенно отвергается.

pressure: среднее 35.74, медиана 35.23, асимметрия 0.73 и эксцесс 2.36, распределение также с правым хвостом. p≈5.65e−19 → далеко от нормального.

vibration: заметно более «тяжёлое» распределение: асимметрия 1.49 и эксцесс 4.40, что говорит о сильной правосторонней асимметрии и пиковости, вероятно из-за редких режимов повышенной вибрации. p≈2.28e−53 → не нормально.

humidity: почти симметричное (асимметрия −0.02), эксцесс 0.82 — распределение ближе к «приплюснутому», но тест всё равно отвергает нормальность (p≈1.53e−3).

faulty: бинарная переменная (0/1), для неё нормальность в принципе неприменима, и KS-тест закономерно даёт p≈0.

Вывод: признаки имеют ненормальные распределения, что типично для сенсорных данных и не препятствует кластеризации. Однако это важно учитывать при выборе масштабирования (лучше StandardScaler или RobustScaler).





Отсутствие классов:
Для кластеризации используются только признаки-описания объекта. Переменная faulty является целевой/контрольной меткой (0 — норма, 1 — неисправность) и не должна включаться в кластеризацию.
Таким образом, условие “отсутствие заранее заданных классов” соблюдается: кластеры строятся без использования меток.

Осмысленность кластеризации:
Кластеризация здесь имеет практический смысл: по показаниям датчиков можно выделить режимы работы оборудования (нормальные и аномальные), а затем проверить, концентрируются ли неисправности в отдельных кластерах. Это соответствует задаче мониторинга промышленного оборудования.

Отсутствие выбросов / работа с выбросами:
По статистике видно наличие “тяжёлых хвостов”, особенно у temperature и vibration (высокий эксцесс и асимметрия). Это означает, что выбросы или экстремальные режимы вероятны.
Для сенсорных данных такие точки часто отражают реальные аварийные состояния, поэтому их не нужно автоматически удалять. Корректный подход:

проверить выбросы визуально (boxplot);

для кластеризации применить устойчивое масштабирование (например, RobustScaler), чтобы выбросы не “перетягивали” расстояния;

при необходимости можно отдельно отметить экстремальные точки как потенциальные аномалии.

Вывод: данные подходят для кластеризации; при этом из-за ненормальности и возможных выбросов нужно использовать нормализацию/масштабирование и интерпретировать экстремальные значения как возможные аномальные режимы.


In [ ]:
import matplotlib.pyplot as plt

num_cols = ["temperature", "pressure", "vibration", "humidity"]

fig, axes = plt.subplots(2, 2, figsize=(10, 6))
axes = axes.ravel()

for ax, col in zip(axes, num_cols):
    ax.boxplot(df[col].dropna(), vert=True)
    ax.set_title(col)
    ax.set_ylabel("Значение")
    ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.suptitle("Boxplot числовых признаков", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()


Судя по твоим boxplot’ам, картина такая:

temperature

Ящик (IQR) узкий вокруг ~65–75, медиана около 70.

Очень много точек выше верхнего уса (примерно >100) и заметно ниже нижнего (примерно <40).

Это означает длинные хвосты и много экстремальных режимов/значений → совпадает с высокой асимметрией и эксцессом в таблице.

pressure

Центр около 30–40, медиана ~35.

Есть выбросы и снизу (очень малые давления), и сверху (до ~80).

Хвосты тоже выраженные, но чуть спокойнее, чем у температуры.

vibration

Основная масса ~1.1–2.0, медиана ~1.5.

Сверху хвост до ~5 — много выбросов.

Есть и отрицательные значения (ниже 0). Для “нормированных единиц” это может быть допустимо (если нормировка относительно нуля), но это точно экстремальные режимы.

humidity

Основная масса ~40–60, медиана ~50.

Выбросы есть, но меньше “катастрофы”, чем в temperature/vibration; хвосты вверх до ~90 и вниз до ~10–20.

По boxplot-графикам выявлены выраженные выбросы в сенсорных признаках, особенно temperature и vibration. Для данных мониторинга оборудования эти экстремальные значения интерпретируются как возможные аномальные режимы работы, поэтому они не удалялись. Для снижения влияния выбросов на расстояния при кластеризации применялось устойчивое масштабирование RobustScaler.

In [ ]:
from sklearn.preprocessing import RobustScaler

num_cols = ["temperature", "pressure", "vibration", "humidity"]

# только числовые признаки
X_num = df[num_cols].copy()

# RobustScaler
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_num)

# обратно в DataFrame для удобства
df_scaled = df.copy()
df_scaled[num_cols] = X_scaled

df_scaled.head()


Для масштабирования числовых признаков выбран метод RobustScaler, так как по boxplot-графикам выявлены выраженные выбросы в признаках temperature и vibration, а также ненормальные распределения всех сенсорных параметров. RobustScaler использует медиану и межквартильный размах (IQR), поэтому устойчив к экстремальным значениям и не позволяет выбросам доминировать в метрике расстояния, что критично для алгоритмов кластеризации на основе расстояний.

In [ ]:
import matplotlib.pyplot as plt
from pandas.plotting import scatter_matrix

# матрица рассеивания на RobustScaler-данных
plt.figure(figsize=(10, 10))
scatter_matrix(
    df_scaled[num_cols],
    figsize=(10, 10),
    diagonal="hist",
    alpha=0.4
)
plt.suptitle("Матрица диаграмм рассеивания (RobustScaler)", y=1.02)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

num_cols = ["temperature", "pressure", "vibration", "humidity"]

X = df_scaled[num_cols].values  # уже RobustScaler!

Ks = range(2, 9)  # обычно смотрят 2..8
inertias = []
silhouettes = []

for k in Ks:
    km = KMeans(n_clusters=k, random_state=42, n_init="auto")
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels))

# --- графики локтя и силуэта на одной картинке ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(Ks, inertias, marker='o')
axes[0].set_title("Метод локтя (Inertia)")
axes[0].set_xlabel("K")
axes[0].set_ylabel("Inertia (сумма квадратов расстояний)")
axes[0].grid(alpha=0.3)

axes[1].plot(Ks, silhouettes, marker='o')
axes[1].set_title("Silhouette score")
axes[1].set_xlabel("K")
axes[1].set_ylabel("Средний силуэт")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# табличкой, чтобы проще было выбрать
for k, iner, sil in zip(Ks, inertias, silhouettes):
    print(f"K={k}: inertia={iner:.2f}, silhouette={sil:.3f}")
